# 02. Data Preprocessing (PySpark)

Notebook ini memuat seluruh logika transformasi *Big Data*. Mulai dari membaca *GeoJSON* dari MongoDB, *flattening*, perbaikan anomali geometri bumi (Trigonometri 3D), hingga *Feature Engineering* yang siap dipakai oleh algoritma K-Means.

### Tahap 1: Import Library & Load Konfigurasi
Menyiapkan seluruh alat dari *PySpark ML* dan fungsi SQL, serta membaca pengaturan `.env`.

In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, split, cos, sin, radians, log1p
from pyspark.sql.types import DoubleType, TimestampType
from pyspark.ml.feature import VectorAssembler, StandardScaler

load_dotenv('../.env')
SPARK_MASTER = os.getenv('SPARK_MASTER_URL', 'local[*]')
MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017')
MONGO_DB = os.getenv('MONGO_DB', 'earthquake_db')
MONGO_RAW_COL = os.getenv('MONGO_RAW_COLLECTION', 'raw_earthquakes')
MONGO_CLEAN_COL = os.getenv('MONGO_CLEAN_COLLECTION', 'clean_earthquakes')
FEATURE_COLS = os.getenv('FEATURE_COLS', 'x,y,z,depth_log,mag').split(',')

### Tahap 2: Menyalakan Spark Session & Konektor MongoDB
Membentuk sesi komputasi terdistribusi dan secara spesifik menyuntikkan *library* konektor MongoDB versi 10.3.0 agar kompatibel dengan PySpark 3.5.0.

In [ ]:
spark = SparkSession.builder \
    .appName("EarthquakePreprocessing") \
    .master(SPARK_MASTER) \
    .config("spark.mongodb.read.connection.uri", MONGO_URI) \
    .config("spark.mongodb.write.connection.uri", MONGO_URI) \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Session Aktif di Master URL: {SPARK_MASTER}")

### Tahap 3: Membaca Data Mentah
Menyedot seluruh dokumen dari MongoDB (*Collection: raw_earthquakes*) dan memuatnya ke dalam memori DataFrame Spark.

In [ ]:
df_raw = spark.read.format("mongodb") \
    .option("database", MONGO_DB) \
    .option("collection", MONGO_RAW_COL) \
    .load()

print(f"Total Data Mentah: {df_raw.count()} baris")
df_raw.printSchema()

### Tahap 4: Flattening (Membuka Nested GeoJSON)
Struktur asli USGS berupa *JSON bersarang* (misal: `geometry.coordinates[0]`). Kita harus mengeluarkannya menjadi baris dan kolom yang datar (*tabular*).

In [ ]:
df_clean = df_raw.select(
    (col("properties.time") / 1000).cast(TimestampType()).alias("time"),
    col("geometry.coordinates")[1].cast(DoubleType()).alias("latitude"),
    col("geometry.coordinates")[0].cast(DoubleType()).alias("longitude"),
    col("geometry.coordinates")[2].cast(DoubleType()).alias("depth"),
    col("properties.mag").cast(DoubleType()).alias("mag"),
    col("properties.place").alias("place"),
    col("properties.type").alias("type")
)

### Tahap 5: Penyaringan (*Filtering*) & Penghapusan Outliers
Hanya mengambil kejadian berjenis `earthquake`. Menghapus baris yang kosong (`null`), menghapus duplikat, dan membuang koordinat yang tidak rasional (misal nilai magnitudo > 10).

In [ ]:
df_clean = df_clean.filter(col("type") == "earthquake").drop("type")
df_clean = df_clean.dropna(subset=["latitude", "longitude", "depth", "mag"])
df_clean = df_clean.dropDuplicates()

df_clean = df_clean.filter((col("depth") >= 0) & (col("depth") <= 700))
df_clean = df_clean.filter((col("mag") >= -2) & (col("mag") <= 10))
df_clean = df_clean.filter((col("latitude") >= -90) & (col("latitude") <= 90))
df_clean = df_clean.filter((col("longitude") >= -180) & (col("longitude") <= 180))

### Tahap 6: Ekstraksi Nama Negara
Mencacah teks panjang di kolom `place` (contoh: *10 km W of Jakarta, Indonesia*) lalu mengambil kata terakhirnya menjadi fitur baru bernama `country`.

In [ ]:
from pyspark.sql.functions import length, when

def extract_country(place_col):
    return trim(split(place_col, ",").getItem(-1))

df_clean = df_clean.withColumn("raw_country", extract_country(col("place")))

us_states = [
    "Alaska", "California", "Hawaii", "Nevada", "Texas", "Washington",
    "Oregon", "Idaho", "Montana", "Wyoming", "Utah", "Colorado", "New Mexico",
    "Arizona", "Oklahoma", "Kansas", "Nebraska", "South Dakota", "North Dakota",
    "Puerto Rico", "U.S. Virgin Islands", "Northern Mariana Islands", "Guam"
]

df_clean = df_clean.withColumn("country",
    when(length(col("raw_country")) == 2, "United States")
    .when(col("raw_country").isin(us_states), "United States")
    .otherwise(col("raw_country"))
).drop("raw_country")

### Tahap 7: Transformasi Spasial 3D (Anti-Distorsi) & Skewness Log
- **Depth Log:** Meratakan kelengkungan ekstrem dari sebaran data kedalaman gempa.
- **Cartesian XYZ:** Mengonversi Koordinat (Bujur/Lintang) menjadi bentuk geometri tiga dimensi (3D). Ini mencegah kesalahan klastering K-Means yang mengira titik koordinat `-179` sangat jauh dengan `+179`.

In [ ]:
df_clean = df_clean.withColumn("depth_log", log1p(col("depth")))

df_clean = df_clean.withColumn("lat_rad", radians(col("latitude"))) \
                   .withColumn("lon_rad", radians(col("longitude"))) \
                   .withColumn("x", cos(col("lat_rad")) * cos(col("lon_rad"))) \
                   .withColumn("y", cos(col("lat_rad")) * sin(col("lon_rad"))) \
                   .withColumn("z", sin(col("lat_rad"))) \
                   .drop("lat_rad", "lon_rad")

print(f"Sisa Data Bersih: {df_clean.count()} baris")

### Tahap 8: Feature Engineering (StandardScaler)
Menggabungkan seluruh kolom angka menjadi sebuah `Vector` raksasa, lalu menskalakan semua nilainya dengan **StandardScaler** agar fitur Kedalaman (ratusan) tidak membayangi fitur Magnitudo (satuan).

In [ ]:
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="raw_features")
df_featured = assembler.transform(df_clean)

scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(df_featured)
df_final = scaler_model.transform(df_featured)

df_final.select("country", "scaled_features").show(5, truncate=False)

### Tahap 9: Menyimpan ke MongoDB
Hasil akhirnya disimpan di koleksi `clean_earthquakes`. Secara _default_ kita atur `mode("overwrite")` agar data lama terhapus jika Anda mengulang kode ini dari awal.

In [ ]:
df_final.write.format("mongodb") \
    .mode("overwrite") \
    .option("database", MONGO_DB) \
    .option("collection", MONGO_CLEAN_COL) \
    .save()

print("Data Sukses Disimpan ke MongoDB Clean Collection!")